# Week 1 — reproduction gate (protocol §A.4)

This notebook contains **no logic**. It imports from `src/` and calls one function
(CLAUDE.md §4). Anything you are tempted to write here belongs in `src/` where it
can be tested.

Attach two Kaggle Datasets before running: one holding this repository, one holding
`mnist.npz` from `scripts/prepare_data.py`. **Their names do not matter** — the next
cell locates them by searching for `src/train.py` and `mnist.npz` anywhere under
`/kaggle/input`, so whatever nesting Kaggle applies to the upload is handled.

Build the code dataset with `python scripts/package_for_kaggle.py`, which excludes
`__pycache__`, `runs/` and `data/`. Uploading the working tree raw trips Kaggle's
1000-file limit and its reserved-name check on `__pycache__`.

Set the accelerator to **GPU T4 x2**. `/kaggle/working` does not persist: run the
last cell before the session ends, and push `runs.zip` to a versioned Dataset.

In [ ]:
import os, sys, glob, shutil, subprocess
from pathlib import Path

# Locate the attached datasets by CONTENT, not by name. Kaggle slugifies dataset
# titles and may nest an upload a level or two deeper than you expect; two
# sessions were lost to exactly that.
_repo_hits = sorted(glob.glob("/kaggle/input/**/src/train.py", recursive=True))
assert _repo_hits, "code dataset not attached (no src/train.py under /kaggle/input)"
REPO = str(Path(_repo_hits[0]).parents[1])

# MNIST ships inside the code dataset; CIFAR-10 is attached separately and comes
# in whichever layout the public dataset happens to use -- src/data.py reads all
# three (npz, extracted pickle batches, or the official tarball).
_mnist = sorted(glob.glob("/kaggle/input/**/mnist.npz", recursive=True))
_cifar = sorted(
    glob.glob("/kaggle/input/**/cifar10.npz", recursive=True)
    + glob.glob("/kaggle/input/**/cifar-10-batches-py", recursive=True)
    + glob.glob("/kaggle/input/**/cifar-10-python.tar.gz", recursive=True)
)
DATA_MNIST = str(Path(_mnist[0]).parent) if _mnist else None
DATA_CIFAR = str(Path(_cifar[0]).parent) if _cifar else None
DATA = DATA_MNIST  # the next cell overrides this for CIFAR experiments
RUNS = "/kaggle/working/runs"

# Must be set before any CUDA context exists (see src/config.set_determinism).
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
sys.path.insert(0, REPO)

import torch
print("REPO       ", REPO)
print("DATA_MNIST ", DATA_MNIST)
print("DATA_CIFAR ", DATA_CIFAR)
print(torch.__version__, torch.cuda.device_count(), "GPU(s)")

## Always run the tests first

Twenty seconds of testing beats losing a 13-hour sweep (CLAUDE.md §8). This runs
on synthetic data and needs no GPU.

In [ ]:
# -p no:cacheprovider: the code dataset is mounted read-only, so pytest cannot
# write .pytest_cache and would fail on that alone.
subprocess.run([sys.executable, "-m", "pytest", "-p", "no:cacheprovider", "tests/"],
               cwd=REPO, check=True)

## Pick the experiment, then smoke-test one config

`EXPERIMENT` selects a directory under `configs/`. The protocol's gate-failure
responses must be tried **in order** (§A.4): `gate_hi` (raise the learning rate),
then `gate_long` (400 tasks), then `gate_narrow` (width 200).

Expect ~14 min per 200-task run on a T4.

In [ ]:
from src.config import load_config
from src.train import Trainer

# --- THE ONLY BLOCK TO EDIT ------------------------------------------------
EXPERIMENT = "gate_hi"
CONFIG_GLOBS = ["*.json"]   # narrow this to split one sweep across accounts
RUN_PATTERN = "gatehi_*"    # which run_ids the evaluation cell reads
IS_GATE = True              # run the frozen gate criterion at the end?
# ---------------------------------------------------------------------------

configs = sorted(
    {p for g in CONFIG_GLOBS for p in glob.glob(f"{REPO}/configs/{EXPERIMENT}/{g}")}
)
assert configs, f"no configs matching {CONFIG_GLOBS} in {REPO}/configs/{EXPERIMENT}"
print(f"{EXPERIMENT}: {len(configs)} configs   <-- CHECK THIS NUMBER BEFORE PROCEEDING")

cfg = load_config(configs[0])
cfg["data"]["root"] = DATA
print(Trainer(cfg, runs_root=RUNS).run())

## The full gate: 15 runs, two per GPU pass

`launch_pair.py` runs two configs at a time via `CUDA_VISIBLE_DEVICES` and stops
launching new work at `--budget-hours`, so the 12-hour kill never lands mid-run.
Anything interrupted resumes from its own checkpoint by `run_id`.

In [ ]:
subprocess.run(
    [sys.executable, "scripts/launch_pair.py", *configs,
     "--runs-root", RUNS, "--data-root", DATA, "--budget-hours", "10.5"],
    cwd=REPO, check=True,
)

## Evaluate the gate

Applies the frozen criterion in `configs/analysis_plan.json`. If the accuracy
gate passes but dead units stay flat, **stop and report it** — that dissociation
is itself a result and changes the paper (protocol §A.4).

In [ ]:
if IS_GATE:
    subprocess.run(
        [sys.executable, "-m", "src.analysis.gate", "--runs-root", RUNS,
         "--pattern", RUN_PATTERN],
        cwd=REPO, check=False,
    )
else:
    print(f"{EXPERIMENT} is not a gate experiment; the frozen gate criterion does "
          "not apply to it. Analysis happens off-Kaggle from extract.zip.")

## Persist results

Three artifacts, increasing in size. All stay attached to this notebook version
on Kaggle, so nothing is lost by downloading them later.

| file | size | what it is | download? |
|---|---|---|---|
| **`extract.zip`** | a few MB | per-task + per-layer metrics, and the recycled-set composition table | **always** — this is C1, C2, C3 and the gate |
| **`c4.zip`** | ~5 MB/run | slim per-neuron table: death flag, magnitude, recycling flag, weight and gradient norms | when doing the C4 survival analysis |
| **`runs.zip`** | ~27 MB/run | everything, incl. the full 23-column per-neuron log and checkpoints | archival only — push to a versioned Dataset |

`c4.zip` omits `sokar_score`: it was 30% of the file and is exactly recomputable
from `mean_abs_act` via `src.analysis.load.add_sokar_score`.

In [ ]:
subprocess.run([sys.executable, "scripts/update_ledger.py", "--runs-root", RUNS,
                "--out", "/kaggle/working/LEDGER.md"], cwd=REPO, check=True)

# 1. Small -- always download this one.
subprocess.run([sys.executable, "scripts/make_analysis_extract.py",
                "--runs-root", RUNS, "--out", "/kaggle/working/extract", "--zip"],
               cwd=REPO, check=True)

# 2. Medium -- the C4 survival dataset, as its OWN zip so it never bloats (1).
subprocess.run([sys.executable, "scripts/make_analysis_extract.py",
                "--runs-root", RUNS, "--out", "/kaggle/working/c4",
                "--with-c4", "--zip"],
               cwd=REPO, check=True)

# 3. Large -- archival. Push to a versioned Dataset; do not download.
print(shutil.make_archive("/kaggle/working/runs", "zip", RUNS))